# 02 - Gender-Role Attitude Scale Construction and Reliability

## Purpose

The purpose of this notebook is to prepare the traditional gender-role attitude measure used in the analysis.

Our study will be using **Q28–Q31** to measure traditional gender-role attitudes.

However, currently, all four statements and options lead to a lower score meaning more traditional gender-role attitudes and a higher score meaning less traditional gender-role attitudes. Therefore, we decided to reverse-code these items so that higher scores indicate more traditional gender-role attitudes, making our final scale much easier to interpret. Furthermore, we will be assessing their internal consistency using Cronbach's alpha before they are combined into a single scale. This is important as the items we are using are not from an established questionnaire scale, so assessing their internal consistency helps us determine whether the four items are consistently measuring traditional gender-role attitudes and can therefore be appropriately combined into a single scale.

## 1. Setup and Data Loading

This section imports the packages required for the analysis and loads the cleaned Singapore World Values Survey dataset produced during the data-cleaning stage.

In [4]:
from pathlib import Path

import pandas as pd

In [ ]:
# Identify the root folder of the GitHub repository.
# This allows the notebook to work whether it is launched from the repository root or from inside the /code folder.

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "code":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "WVS_Singapore_cleaned.csv"
)

# Load the cleaned dataset
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print(f"Number of respondents: {df.shape[0]}")
print(f"Number of variables: {df.shape[1]}")

Dataset loaded successfully.
Number of respondents: 2012
Number of variables: 15


## 2. Inspecting the Gender-Role Attitude Items

Before making any changes, we decided to inspect Q28–Q31 to confirm that their values are coded as expected.
According to the coding document, the original response scale is:

- **1 = Strongly agree**
- **2 = Agree**
- **3 = Disagree**
- **4 = Strongly disagree**

In [6]:
gender_items = ["Q28", "Q29", "Q30", "Q31"]

df[gender_items].head()

,Q28,Q29,Q30,Q31
0,4.0,1.0,4.0,2.0
1,2.0,1.0,1.0,1.0
2,3.0,4.0,4.0,3.0
3,3.0,4.0,4.0,4.0
4,1.0,4.0,4.0,4.0


In [7]:
# Check all observed values for each item

for item in gender_items:
    values = sorted(df[item].dropna().unique())
    print(f"{item}: {values}")

Q28: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Q29: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Q30: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]
Q31: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0)]


In [8]:
df[gender_items].isna().sum()

Q28     8
Q29    24
Q30     7
Q31    13
dtype: int64

This helped to confirm that all four items (Q28–Q31) contained the expected values of 1 to 4, with no unexpected values. However, it also helped to alert us that there were some missing responses, with 8 for Q28, 24 for Q29, 7 for Q30, and 13 for Q31.

## 3. Reverse Coding Q28–Q31

In the original coding, lower scores indicate higher traditional gender-role attitudes.

To make the final scale easier to interpret, we decided to reverse-code Q28–Q31 so that:

- **1 = Less traditional gender-role attitudes**
- **4 = More traditional gender-role attitudes**

In [9]:
# Create names for the reverse-coded variables
reversed_items = [f"{item}_rev" for item in gender_items]

# Reverse-code Q28–Q31
df[reversed_items] = 5 - df[gender_items]

In [11]:
#Check that the reverse coding worked
df[
    [
        "Q28", "Q28_rev",
        "Q29", "Q29_rev",
        "Q30", "Q30_rev",
        "Q31", "Q31_rev"
    ]
].head(10)

,Q28,Q28_rev,Q29,Q29_rev,Q30,Q30_rev,Q31,Q31_rev
0,4.0,1.0,1.0,4.0,4.0,1.0,2.0,3.0
1,2.0,3.0,1.0,4.0,1.0,4.0,1.0,4.0
2,3.0,2.0,4.0,1.0,4.0,1.0,3.0,2.0
3,3.0,2.0,4.0,1.0,4.0,1.0,4.0,1.0
4,1.0,4.0,4.0,1.0,4.0,1.0,4.0,1.0
5,3.0,2.0,4.0,1.0,4.0,1.0,4.0,1.0
6,3.0,2.0,2.0,3.0,2.0,3.0,2.0,3.0
7,4.0,1.0,4.0,1.0,4.0,1.0,4.0,1.0
8,3.0,2.0,3.0,2.0,4.0,1.0,3.0,2.0
9,3.0,2.0,NaN,NaN,3.0,2.0,4.0,1.0


## 4. Internal Consistency: Cronbach's Alpha

Before combining Q28–Q31 into a single scale, Cronbach's alpha is calculated to assess their internal consistency and to ensure that they all measure traditional gender-role attitudes.

For this reliability analysis, we only include respondents who provided valid responses to all four items.

In [12]:
gender_scale_data = df[reversed_items].dropna()

print(
    f"Complete cases used for Cronbach's alpha: "
    f"{len(gender_scale_data)}"
)

Complete cases used for Cronbach's alpha: 1970


In [13]:
def cronbach_alpha(items):
    """
    Calculate Cronbach's alpha for a set of questionnaire items.

    Parameters
    ----------
    items : pandas.DataFrame
        Each column represents one scale item.

    Returns
    -------
    float
        Cronbach's alpha.
    """

    # Number of items
    k = items.shape[1]

    # Variance of each item
    item_variances = items.var(axis=0, ddof=1)

    # Total score for each respondent
    total_scores = items.sum(axis=1)

    # Variance of total scores
    total_variance = total_scores.var(ddof=1)

    # Cronbach's alpha
    alpha = (
        k / (k - 1)
        * (1 - item_variances.sum() / total_variance)
    )

    return alpha

In [14]:
alpha = cronbach_alpha(gender_scale_data)

print(f"Cronbach's alpha = {alpha:.3f}")

Cronbach's alpha = 0.767


In [15]:
alpha_original = cronbach_alpha(
    df[gender_items].dropna()
)

alpha_reversed = cronbach_alpha(
    df[reversed_items].dropna()
)

print(f"Original coding: α = {alpha_original:.3f}")
print(f"Reverse coding:  α = {alpha_reversed:.3f}")

Original coding: α = 0.767
Reverse coding:  α = 0.767


### Interpretation

The Cronbach's alpha that we obtained was approximately **α = .767.** This indicates that Q28–Q31 have sufficient internal consistency to be combined into a composite measure of traditional gender-role attitudes. However, to make sure that the reverse coding we did did not affect the Cronbach's alpha, we double-checked by running the Cronbach's alpha using the original item coding as well. It showed the same Cronbach's alpha, so we know that the reverse coding did not affect the reliability of the scale.

## 5. Constructing the Traditional Gender-Role Attitude Scale

Since we now know that the reverse-coded Q28–Q31 items can be combined into a composite measure of traditional gender-role attitudes, the composite score is calculated as the mean of the four items, with higher scores representing more traditional gender-role attitudes.

We only calculated a composite score for respondents who had valid responses to all four items, since the reliability analysis was conducted using the same criteria. This ensures that missing responses do not affect the internal consistency and reliability of the composite measure.

In [ ]:
df["traditional_gender_attitudes"] = df[
    reversed_items
].mean(
    axis=1,
    skipna=False
)

In [ ]:
#Check whether it worked
df[
    reversed_items + ["traditional_gender_attitudes"]
].head(10)

,Q28_rev,Q29_rev,Q30_rev,Q31_rev,traditional_gender_attitudes
0,1.0,4.0,1.0,3.0,2.25
1,3.0,4.0,4.0,4.0,3.75
2,2.0,1.0,1.0,2.0,1.50
3,2.0,1.0,1.0,1.0,1.25
4,4.0,1.0,1.0,1.0,1.75
5,2.0,1.0,1.0,1.0,1.25
6,2.0,3.0,3.0,3.0,2.75
7,1.0,1.0,1.0,1.0,1.00
8,2.0,2.0,1.0,2.0,1.75
9,2.0,NaN,2.0,1.0,NaN


## 6. Saving the Scored Dataset
Lastly, we saved the reverse-coded items and composite traditional gender-role attitude score as a new processed dataset file. 



In [19]:
SCORED_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "WVS_Singapore_scored.csv"
)

df.to_csv(
    SCORED_DATA_PATH,
    index=False
)

print(f"Scored dataset saved to:\n{SCORED_DATA_PATH}")

Scored dataset saved to:
/Users/joellelim/Desktop/HS4002/Group Project/hs4002-labrador-2026/data/processed/WVS_Singapore_scored.csv
